# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all available record sets in the dataset (referenced by their @id fields)
def describe_record_sets(dataset):
    print("Available Record Sets (by @id):")
    record_sets = []
    for rset in dataset.record_sets:
        print(f"  - @id: {rset['@id']}")
        print(f"    Name: {rset.get('name', '')}")
        print(f"    Description: {rset.get('description', '')}")
        record_sets.append(rset['@id'])
        # List the fields for each record set
        print("    Fields and their @id's:")
        for field in rset.get('field', []):
            if isinstance(field, dict):
                print(f"       - @id: {field['@id']} | name: {field.get('name', '')}")
            else:
                # If field is just a string @id
                print(f"       - @id: {field}")
    return record_sets

record_sets = describe_record_sets(dataset)

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# For demonstration, let's use the first record set available
if len(record_sets) == 0:
    raise ValueError("No record sets found in the dataset.")
else:
    selected_record_set_id = record_sets[0]

# Load all records for each record set into DataFrames by @id
dataframes = {}
for rset_id in record_sets:
    # Using the `records` generator
    records = list(dataset.records(record_set=rset_id))
    df = pd.DataFrame(records)
    dataframes[rset_id] = df
    print(f"Loaded {len(df)} records for record set '{rset_id}'")

print(f"\nColumns in DataFrame for record set '{selected_record_set_id}':")
print(dataframes[selected_record_set_id].columns.tolist())
dataframes[selected_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Inspect DataFrame to select numeric fields for analysis
df = dataframes[selected_record_set_id]
print("Numeric columns detected:")
numeric_fields = df.select_dtypes(include='number').columns.tolist()
print(numeric_fields)

if len(numeric_fields) == 0:
    print("No numeric fields available for EDA.")
else:
    # Use the first numeric field for demonstration
    numeric_field = numeric_fields[0]
    threshold = df[numeric_field].mean()
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    print(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping by a categorical (non-numeric) field
    categorical_fields = df.select_dtypes(include='object').columns.tolist()
    group_field = None
    for col in categorical_fields:
        if df[col].nunique() < len(df) // 2:
            group_field = col
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
        print(f"\nGrouped {numeric_field} by {group_field} (mean):")
        print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if len(numeric_fields) > 0:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of '{numeric_field}' in record set '{selected_record_set_id}'")
    plt.xlabel(numeric_field)
    plt.show()
    
    # If a group_field exists, show boxplot
    if group_field:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"Boxplot of '{numeric_field}' grouped by '{group_field}'")
        plt.xticks(rotation=30)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

*This notebook demonstrated how to load and inspect the FAIR^2 mass spectrometry dataset via `mlcroissant`, overview its schema via record set and field `@id` references, extract and analyze tabular data, and perform simple exploratory data analysis and visualization. For further use, reference the schema and the `mlcroissant` documentation to access additional fields or complex relationships within the dataset.*